In [ ]:
%pip install numpy
%pip install scikit-learn
%pip install scipy
%pip install geopandas rasterio rasterstats shapely
!pip install hmmlearn          

In [ ]:
from hmmlearn import base, hmm
import geopandas as gpd
import pandas as pd
import shapely as shp
import matplotlib.pyplot as plt
import numpy as np
import rasterio
from rasterio.features import rasterize
from rasterstats.io import bounds_window
import rasterstats

In [ ]:
from os import path as op
import pickle
from google.colab import files
import matplotlib.pyplot as plt

In [ ]:
from google.colab import drive
drive.mount('/content/drive/', force_remount=True)

Mounted at /content/drive/


In [ ]:
import os
# list the folders and files in your drive
mydrive = '/content/drive/My Drive'
colab = '/content/drive/My Drive/Colab Notebooks/Ground Cover Model'
print(os.listdir(mydrive))
# list the current working directory
print(os.getcwd())
# make a data folder in your Colab Notebook directory if it doesn't exist
data_folder = os.path.join(colab,'data')
os.path.isdir(data_folder)
if (not os.path.isdir(data_folder)):
  os.mkdir(data_folder)
# list the contents
print(os.listdir(colab))

['CONTRATO DISEÑO ARQUITECTONICO - POSEIDON.doc', 'Master Investigation', 'Domun', 'escala 18.xlsx', 'Relacion de Precios.gsheet', 'Puchi', 'Time Table.xlsx', 'Time Table.pdf', 'Carte-Recomendación-MEXT-2 (1).doc', 'Carte-Recomendación-MEXT-2 (1).gdoc', 'Copia de Template: Initial Design Presentation to Experts.gslides', '04申請書ファイルＢ_-format A.gdoc', 'スクリプト_マルティンガルシア.docx', 'スクリプト_マルティンガルシア.gdoc', 'Competitions', 'Architecture Projects', 'Experimenting With Architectural Compe..gslides', 'Windows in Art.gslides', 'Visit to the 2011 GEJE Recovered Cities.gslides', 'Urban recovery Pisco 2007 Peru.gslides', 'EcoModel City Toyama.gslides', 'Report_Igarashi.gdoc', 'aitcd-B9TM7016.gdoc', 'TITULO.gdoc', '01 IMG_20191002_163425_897.jpg', '02 .jpg', '03.jpg', '04.jpg', '05 (1).jpg', '06IMG_20191002_163407_010.jpg', '08.jpg', '09.jpg', 'The Great Kanto Earthquake of 1923.gslides', 'Segmentation LATIR.gsheet', 'Q&A 1923 Great Kanto Earthquake.gslides', 'Earthquake Disaster Control 2019.gsli

In [ ]:
my_root_dir = "/content/drive/My Drive/Colab Notebooks/Ground Cover Model/data"

In [ ]:
lc_probas_yr01 = '/content/drive/My Drive/Colab Notebooks/Ground Cover Model/data/lc_probas_yr01.tif'
lc_probas_yr02 = '/content/drive/My Drive/Colab Notebooks/Ground Cover Model/data/lc_probas_yr02.tif'
season_01 = '/content/drive/My Drive/Colab Notebooks/Ground Cover Model/data/ch_01.tif'
print(lc_probas_yr01)
print(lc_probas_yr02)
print(season_01)

/content/drive/My Drive/Colab Notebooks/Ground Cover Model/data/lc_probas_yr01.tif
/content/drive/My Drive/Colab Notebooks/Ground Cover Model/data/lc_probas_yr02.tif
/content/drive/My Drive/Colab Notebooks/Ground Cover Model/data/ch_01.tif


In [ ]:

src = rasterio.open(lc_probas_yr01, 'r')
profile = src.profile
profile.update(
    dtype=rasterio.uint8,
    count=1,
    nodata=10)

# perform prediction on each small image patch to minimize required memory
patch_size = 500

for i in range((src.shape[0] // patch_size) + 1):
    for j in range((src.shape[1] // patch_size) + 1):
        # define the pixels to read (and write)
        window = rasterio.windows.Window(
            j * patch_size,
            i * patch_size,
            # don't read past the image bounds
            min(patch_size, src.shape[1] - j * patch_size),
            min(patch_size, src.shape[0] - i * patch_size)
        )
        
        data = src.read(window=window)
        # read the image into the proper format, adding indices if necessary
        img_swp_1 = np.moveaxis(data, 0, 2)
        img_flat_1 = img_swp_1.reshape(-1, img_swp_1.shape[-1])
        z = abs(img_flat_1)
        X1 = np.array(np.where(z > 1, 10, z), int)
        #X1 = X1[X1 != 10]
src.close()


In [ ]:
X1.shape

(85646, 1)

In [ ]:

src = rasterio.open(lc_probas_yr02, 'r')
profile = src.profile
profile.update(
    dtype=rasterio.int8,
    count=1,
    nodata=10)

# perform prediction on each small image patch to minimize required memory
patch_size = 500

for i in range((src.shape[0] // patch_size) + 1):
    for j in range((src.shape[1] // patch_size) + 1):
        # define the pixels to read (and write)
        window = rasterio.windows.Window(
            j * patch_size,
            i * patch_size,
            # don't read past the image bounds
            min(patch_size, src.shape[1] - j * patch_size),
            min(patch_size, src.shape[0] - i * patch_size)
        )
        
        data = src.read(window=window)
        # read the image into the proper format, adding indices if necessary
        img_swp_2 = np.moveaxis(data, 0, 2)
        img_flat_2 = img_swp_2.reshape(-1, img_swp_2.shape[-1])
        l = abs(img_flat_2)
        X2 = np.array(np.where(l > 1, 10, l), int)
        #X2 = X2[X2 != 10]
src.close()

In [ ]:
X2.shape

(85646, 1)

In [ ]:

non=10
src = rasterio.open(season_01, 'r')
profile = src.profile
profile.update(
    dtype=rasterio.int8,
    count=1,
    nodata=non)

# perform prediction on each small image patch to minimize required memory
patch_size = 500

for i in range((src.shape[0] // patch_size) + 1):
    for j in range((src.shape[1] // patch_size) + 1):
        # define the pixels to read (and write)
        window = rasterio.windows.Window(
            j * patch_size,
            i * patch_size,
            # don't read past the image bounds
            min(patch_size, src.shape[1] - j * patch_size),
            min(patch_size, src.shape[0] - i * patch_size)
        )
        
        data = src.read(window=window)
        # read the image into the proper format, adding indices if necessary
        img_swp_3 = np.moveaxis(data, 0, 2)
        img_flat_3 = img_swp_3.reshape(-1, img_swp_3.shape[-1])
        t = [i for i in img_flat_3+1]
        d = np.array(t, int)
        X3 = np.array(np.where(d < -1, 10, d), int)
        #X3 = X3[X3 != 10]

src.close()


In [ ]:
X3.shape
np.unique(X3)

array([ 0,  1,  2, 10])

In [ ]:
#Hidden Markov Model_EM 'Forward-Backward' Algorithm
from hmmlearn import hmm
from hmmlearn.base import ConvergenceMonitor
from hmmlearn.hmm import MultinomialHMM
from math import floor

new_image = lc_probas_yr02

src = rasterio.open(new_image, 'r')
profile = src.profile
profile.update(
    dtype=rasterio.int8,
    count=1,
    nodata=101)

dst = rasterio.open(new_image, 'w', **profile)

# perform prediction on each small image patch to minimize required memory
patch_size = 500

for i in range((src.shape[0] // patch_size) + 1):
    for j in range((src.shape[1] // patch_size) + 1):
        # define the pixels to read (and write)
        window = rasterio.windows.Window(
            j * patch_size,
            i * patch_size,
            # don't read past the image bounds
            min(patch_size, src.shape[1] - j * patch_size),
            min(patch_size, src.shape[0] - i * patch_size)
        )
        
        data = src.read(window=window)
        # read the image into the proper format, adding indices if necessary
        img_swp = np.moveaxis(data, 0, 2)
        img_flat = img_swp.reshape(-1, img_swp.shape[-1])
        states = ['loss', 'no change', 'gain']
        labels_hmm = dict(zip(range(len(states)), states))
        lengths = [len(X1), len(X2)]

        X = np.concatenate(([X1,X2])).reshape(1, -1)
        #XX = np.random.choice(X, size=171292).reshape(-1,1)

        model = hmm.MultinomialHMM(n_components=2)   
        model.n_iter =800
        model.tol = .01
        model.verbose = False
        class ThresholdMonitor(ConvergenceMonitor):
            @property
            def converged(self):
                return (self.iter == self.n_iter or
                        self.history[-1] >= self.tol)

        model.monitor_ = ThresholdMonitor(model.n_iter, model.tol, model.verbose)
        model.fit(X)
        g = model.predict_proba(X3)
        
        m = np.ma.masked_invalid(img_flat)
        
        out_prob = []
        b = np.array([1])
        prob = np.take_along_axis(g, b[:,None], axis=1)

        for index_1, value in enumerate(X3):

              if prob[index_1] < 0.5 and value==0:
                w = value+3
                out_prob.append(w)
              elif value==1 or value==0 or value==10:
                w = value*0
                out_prob.append(w)
              elif prob[index_1] > 0.5 and value==2:
                w = value+3
                out_prob.append(w)
              elif prob[index_1] < 0.5 and value==2:
                w = value-2
                out_prob.append(w)
              
        img_preds = np.array(out_prob).reshape(-1, 1)
        output = np.zeros(img_flat.shape[0])
        
        # add the prediction back to valid pixels and fill blanks with zeros
        output[~m.mask[:,0]] = img_preds.flatten()
        output = output.reshape(*img_swp.shape[:-1])
        
        # create final mask
        mask = (~m.mask[:,0]).reshape(*img_swp.shape[:-1])

        # write the final file
        dst.write(output.astype(rasterio.int8), 1, window=window)
        dst.write_mask(mask, window=window)
        # write the final file
        dst.write(output.astype(rasterio.int8), 1, window=window)
        dst.write_mask(mask, window=window)

src.close()
dst.close()

In [ ]:
#img_preds[1000:2000,]
#g[34123:34151]
#print([labels_hmm[x] for x in img_preds])
#img_preds[54123:54151]
#prob[54123:54151]
#np.unique(out_prob)
#X3[54123:54151]
#[k for k in prob[54123:54151] > 0.5]
#out_prob[54123:54151]
#g[54123:54151]
#out_prob
#len(out_prob)
#X3[54123:54151]
#img_preds[54123:54151]
#np.unique(img_preds)
g




In [ ]:
print(model.transmat_)

print('Original and learned observation probabilities')
print(model.emissionprob_)